In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [26]:
def calculate_egfr(creatinine, age=50, gender='male'):
    if gender.lower() == 'female':
        return 175 * (creatinine ** -1.154) * (age ** -0.203) * 0.742
    else:
        return 175 * (creatinine ** -1.154) * (age ** -0.203)

In [27]:
def get_ckd_stage(egfr):
    if egfr >= 90:
        return "Stage 1 (Normal)"
    elif egfr >= 60:
        return "Stage 2 (Mild)"
    elif egfr >= 30:
        return "Stage 3 (Moderate)"
    elif egfr >= 15:
        return "Stage 4 (Severe)"
    else:
        return "Stage 5 (Kidney Failure)"

In [28]:
data = pd.read_csv("kidney_disease.csv")

In [29]:
data.replace({
    'normal': 1, 'abnormal': 0,
    'present': 1, 'notpresent': 0,
    'yes': 1, 'no': 0,
    'good': 1, 'poor': 0,
    'ckd': 1, 'notckd': 0
}, inplace=True)

/tmp/ipython-input-2476229414.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({


In [30]:
data = data.apply(pd.to_numeric, errors='coerce')
data.fillna(data.mean(), inplace=True)

In [31]:
data['classification'] = data['classification'].round().astype(int)

In [32]:
X = data.drop(columns=['classification'])
y = data['classification']

In [33]:
leakage_cols = ['sg', 'hemo', 'rc', 'pcv']
X = X.drop(columns=[col for col in leakage_cols if col in X.columns])

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [35]:
model = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, min_samples_split=10, random_state=42)

In [36]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 100.00%

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        38
           1       1.00      1.00      1.00        62

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [37]:
patient_data = pd.DataFrame(columns=X.columns)

In [38]:
patient_data.loc[0, 'sc'] = 1.3
patient_data.loc[0, 'age'] = 55
patient_data.loc[0, 'bp'] = 80
patient_data.loc[0, 'bgr'] = 120

In [39]:
for col in X.columns:
    if pd.isna(patient_data.loc[0, col]):
        patient_data.loc[0, col] = 0

In [45]:
prediction = model.predict(patient_data)

if prediction[0] == 1:
    serum_creatinine = patient_data['sc'].values[0]
    egfr = calculate_egfr(
        serum_creatinine,
        age=patient_data['age'].values[0],
        gender='male'
    )
    stage = get_ckd_stage(egfr)

    print("\nChronic Kidney Disease Detected")
    print(f"Serum Creatinine: {serum_creatinine}")
    print(f"Estimated eGFR: {egfr:.2f}")
    print("CKD Stage:", stage)
else:
    print("\nNo Chronic Kidney Disease Detected")


Chronic Kidney Disease Detected
Serum Creatinine: 1.3
Estimated eGFR: 57.31
CKD Stage: Stage 3 (Moderate)
